In this notebook, I took out the feature extractor (a pre-trained model) from the Data Loader and added it into the Encoder. The name of the notebook also holds this: *_TRansferLEarningInENCoder

In [ ]:
#!pip install numpy
#!pip install pandas
#!pip install matplotlib
#!pip install torch
#!pip install torchvision
!pip install scikit-learn
#!pip install opencv-python
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import (CosineAnnealingLR,
                                      CosineAnnealingWarmRestarts,
                                      StepLR,
                                      ExponentialLR)
import sklearn.utils
from sklearn.model_selection import train_test_split
import cv2

In CeDAR (Center for Data Analytics Research, ADA University) option, the notebook shall connevt to the local machine (where the whole dataset is supposed to be). To connect to the CeDAR's environment run the following to start Jupyter with access:


```
jupyter notebook \
>   --NotebookApp.allow_origin='https://colab.research.google.com' \
>   --port=8888 \
>   --NotebookApp.port_retries=0
```
Then select "Connect to a local runtime" and put the link of notebook environment (from the console)



In [ ]:
class Config:
    debug = False
    env = 'CeDAR' # Dev (Jamal's GoogleDrive), Prod (SLR GDrive) or CeDAR (local)
    csv_path = ''
    seed = 44
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    # device = 'tpu' # uncomment to switch to TPU usage

    video_processing_tool = 'TorchVision' # OpenCV, VidGear or TorchVision
    max_frames = 64
    max_words_in_sentence = 10


    drive_folder = '/home/temporaryuser2/gdrive/SLR/Data' # path for the local (CeDAR)
    if (env == 'Dev'):
      drive_folder = 'drive/MyDrive/SLR_test'
    elif (env == 'Prod'):
      drive_folder = 'drive/MyDrive/SLR/Data'

    video_folder = drive_folder+'/Video'

    train_csv_path = drive_folder + '/sentences_all.csv'
    camera_source = 'Cam2' # Cam1 - side-top, Cam2 - front
    BATCH_SIZE = 1 #updated before the training

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
#    torch.manual_seed(seed)
    # if torch.cuda.is_available():
    #     torch.cuda.manual_seed(seed)

config = Config()
seed_everything(config.seed)
print('Running on',config.device)
if torch.cuda.is_available():
    print('GPU number:',torch.cuda.device_count())

In [ ]:
if (config.device == 'tpu'):
  !pip install cloud-tpu-client==0.10 torch==2.0.0 torchvision==0.15.1 https://storage.googleapis.com/tpu-pytorch/wheels/colab/torch_xla-2.0-cp39-cp39-linux_x86_64.whl

  import torch_xla
  import torch_xla.core.xla_model as xm


  dev = xm.xla_device()
  t1 = torch.ones(3, 3, device = dev)
  print(t1)
  config.device = dev

In [ ]:
import sys
import subprocess

def pip_install(package):
  subprocess.check_call([sys.executable, '-m', 'pip', 'install',package])

In [ ]:
pip_install('mediapipe')

# https://github.com/jbohnslav/opencv_transforms
pip_install('opencv_transforms')

if config.video_processing_tool == 'VidGear':
  pip_install('vidgear[core]')

In [ ]:
pip install openpyxl


In [ ]:
ls

In [ ]:
df = pd.read_csv('/home/temporaryuser2/gdrive/SLR/Data/sentences_all.csv',
                 sep=';',
                 encoding='utf-8')

df.head()


In [ ]:
import os
import mimetypes

path = config.train_csv_path
print("Exists:", os.path.exists(path))
print("Size:", os.path.getsize(path), "bytes")
print("Mime:", mimetypes.guess_type(path))


In [ ]:
train_set_size = 50

import mediapipe as mp
import pandas as pd
import os
import torch

# ----------------------------------------
# 1. LOAD CSV CORRECTLY
# ----------------------------------------
# Your CSV has no header → 3 columns → we rename them manually
sentences = pd.read_csv(
    config.train_csv_path,
    sep=';',
    encoding='utf-8',
    header=None,
    names=['idd', 'sentence', 'sign_language']
)

# Limit to first N samples
sentences = sentences.iloc[:train_set_size, :]


# ----------------------------------------
# 2. BUILD UNIQUE WORD LIST
# ----------------------------------------
word_set = set(['SOS', 'EOS'])

# Use the 3rd column = sign_language
sentences['sign_language'].str.lower().str.split().apply(word_set.update)

sorted_word_set = sorted(word_set)
print("Unique words:", sorted_word_set)


# ----------------------------------------
# 3. CREATE WORD <-> INDEX MAPPINGS
# ----------------------------------------
encodings = {k: v for v, k in enumerate(sorted_word_set)}
word_idx  = {v: k for v, k in encodings.items()}

print("Word encodings:", encodings)
print("Words by index:", word_idx)

# Save dictionaries
torch.save(encodings, os.path.join(config.drive_folder, 'jamal', 'encodings.dict'))
torch.save(word_idx, os.path.join(config.drive_folder, 'jamal', 'word_idx.dict'))


# ----------------------------------------
# 4. SENTENCE TO INDEX ENCODING
# ----------------------------------------
def get_sentence_encoded(sentence):
    tokens = ('SOS ' + sentence + ' EOS').split()
    encoded = [encodings[tok] for tok in tokens]

    # Pad / truncate
    if len(encoded) > config.max_words_in_sentence:
        encoded = encoded[:config.max_words_in_sentence]
    else:
        encoded += [0] * (config.max_words_in_sentence - len(encoded))

    return encoded


if config.debug:
    print(get_sentence_encoded('mən hansı sənəd vermək'))
    print(get_sentence_encoded('mən bakı yaşamaq'))


# ----------------------------------------
# 5. BUILD (video_file, encoding) TABLE
# ----------------------------------------
df = pd.DataFrame(columns=["idd", "video_file", "encoding"])

for _, row in sentences.iterrows():
    sid     = int(row['idd'])
    phrase  = str(row['sign_language']).lower()
    encoded = get_sentence_encoded(phrase)

    # Path example:
    # /content/drive/MyDrive/SLR/Data/Video/Cam2/5/
    dir_path = os.path.join(config.video_folder, config.camera_source, str(sid))

    if not os.path.exists(dir_path):
        print("⚠️ Missing directory:", dir_path)
        continue

    for filename in os.listdir(dir_path):
        full_path = os.path.join(dir_path, filename)
        if os.path.isfile(full_path):
            df.loc[len(df)] = [sid, full_path, encoded]


if config.debug:
    print(df.head())


This small piece of code is to test the outputs of the pre-trained model.

In [ ]:
#if# from torchvision.models.feature_extraction import create_feature_extractor

# x = torch.rand(1, 3, 224, 224).to(config.device)

# return_nodes = {
#     'features.12.cat': 'layer12'
# }
# model_new = create_feature_extractor(model, return_nodes=return_nodes).to(config.device)

# result = model_new(x)
# n,out_filters,out_width,out_height = result['layer12'].shape
# print(n,out_filters,out_width,out_height)

In [ ]:
import math

rows = int(math.sqrt(config.max_frames))
cols = config.max_frames//rows

# ✅ CHANGED: Fixed visualization to handle 5D tensors and normalize pixel values
def visualize_frames(frames):
  # Handle batched input: (B,T,C,H,W) → (T,C,H,W)
  if frames.ndim == 5:
    frames = frames.squeeze(0)


  fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(20,20))

  idx = 0
  for i in range(rows):
      for j in range(cols):
        if idx < frames.shape[0]:
          # Convert to (H,W,C) format
          frame = frames[idx,:,:,:].permute(1,2,0).cpu()

          # Normalize to [0, 1] range for proper display
          frame_min = frame.min()
          frame_max = frame.max()
          if frame_max > frame_min:
            frame_normalized = (frame - frame_min) / (frame_max - frame_min)
          else:
            frame_normalized = frame

          axes[i, j].imshow(frame_normalized)
          axes[i, j].axis('off')
        idx += 1
  plt.tight_layout()
  plt.show()

In [ ]:
pip_install('pytorchvideo')

TODO: Implement augmentation to the data

In [ ]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import torch, torchvision, pytorchvideo
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
import torch
print("cuda ok?:", torch.cuda.is_available())


In [ ]:
!nvidia-smi


In [ ]:
# --- GEÇİCİ ÇÖZÜM: pytorchvideo'nun eski importunu yamala ---

# 1) Shim: functional_tensor modül adını functional'a yönlendir
import sys
from torchvision.transforms import functional as F
sys.modules['torchvision.transforms.functional_tensor'] = F

# 2) Artık importlar çalışmalı
from torchvision import transforms
from pytorchvideo.transforms import ApplyTransformToKey, Normalize

print("✅ Importlar tamam: torchvision & pytorchvideo birlikte çalışıyor.")


In [ ]:
# Modern, clean imports (NO WARNINGS)

import torch
import torchvision
from torchvision import transforms
from torchvision.transforms import (
    Compose,
    Lambda,
    RandomCrop,
    CenterCrop,
    RandomAdjustSharpness,
    Resize,
    ColorJitter,
    RandomHorizontalFlip
)
from torchvision.transforms import functional as F  # official non-deprecated functions

# Video transforms - official and recommended
from pytorchvideo.transforms import (
    ApplyTransformToKey,
    Normalize,
    RandomShortSideScale,
    UniformTemporalSubsample,
    Permute,
)


In [ ]:
def keep_frames_with_hands(video_data, crop_size: int = None,
                           mp_min_detection_confidence: float = 0.8,
                           mp_min_tracking_confidence: float = 0.9):
    """
    Returns video frames (N, 3, H, W) where hands are detected using MediaPipe.
    Prevents channel warnings by enforcing correct shape and dtype.
    Now moves all tensor transforms to GPU as early as possible.
    """

    import numpy as np
    mpHands = mp.solutions.hands
    hands = mpHands.Hands(static_image_mode=True, max_num_hands=2,
                          min_detection_confidence=mp_min_detection_confidence,
                          min_tracking_confidence=mp_min_tracking_confidence)

    # Create output tensor
    if crop_size:
        video_arr = torch.zeros((0, 3, crop_size, crop_size), device=config.device)
        transform = Compose([CenterCrop(crop_size)])
        expected_h, expected_w = crop_size, crop_size
    else:
        expected_h, expected_w = 960, 1280
        video_arr = torch.zeros((0, 3, expected_h, expected_w), device=config.device)

    # Iterate through video frames
    for frame in video_data:
        # Convert tensor → numpy
        if torch.is_tensor(frame):
            # Frame is (C,H,W) or (H,W,C) → we make it (H,W,C)
            if frame.ndim == 3 and frame.shape[0] in [1, 3]:
                frame_np = frame.permute(1, 2, 0).cpu().numpy()
            else:
                frame_np = frame.cpu().numpy()
        else:
            frame_np = frame

        # Ensure correct dtype
        frame_np = np.ascontiguousarray(frame_np).astype(np.uint8)

        # ✅ Fix channel order to ALWAYS RGB, HWC
        # Case 1: (H,W,3) BGR from OpenCV/VidGear
        if config.video_processing_tool in ['OpenCV', 'VidGear']:
            if frame_np.ndim == 3 and frame_np.shape[2] == 3:
                frame_rgb = cv2.cvtColor(frame_np, cv2.COLOR_BGR2RGB)
            else:
                continue

        # Case 2: TorchVision gives (3,H,W) or (T,3,H,W)
        else:
            if frame_np.ndim == 3 and frame_np.shape[0] == 3:
                frame_rgb = np.transpose(frame_np, (1, 2, 0))  # → HWC
            elif frame_np.ndim == 3 and frame_np.shape[2] == 3:
                frame_rgb = frame_np  # already fine
            else:
                print(f"Warning: Unexpected frame shape:{frame_np.shape}")
                continue

        # MediaPipe hand detection
        results_hands = hands.process(frame_rgb)

        if results_hands.multi_hand_landmarks:
            frame_tensor = torch.from_numpy(frame_rgb).permute(2, 0, 1).to(config.device)  # → CHW

            # Apply center crop if required (on GPU)
            if crop_size:
                frame_tensor = transform(frame_tensor.to(config.device))

            # Add to buffer (on GPU)
            video_arr = torch.cat((video_arr, frame_tensor.unsqueeze(0)), dim=0)

    # ✅ If no hand frames detected → return black frames to avoid crashing
    if video_arr.shape[0] == 0:
        print("No frames with hands detected in the video.")
        video_arr = torch.zeros((config.max_frames, 3, expected_h, expected_w), device=config.device)

    return video_arr

TODO: It seems the data is not read randomly. It's read sequentially.

In [ ]:
def apply_video_transforms(resize_size: int = 224):
    video_transform=Compose([
        # Apply Resize to each frame individually
        Lambda(lambda x: torch.stack([transforms.Resize(size=(resize_size, resize_size))(frame) for frame in x])),
        # UniformTemporalSubsample(25),
        #ColorJitter(brightness=0.5, contrast=0.5),
        # RandomShortSideScale(min_size=256, max_size=512),
        #RandomHorizontalFlip(p=0.5),
    ])

    return video_transform

In [ ]:
def pad_collate(batch):
    videos, labels, fnames = zip(*batch)

    # lengths of each video sequence in this batch
    lengths = [v.shape[0] for v in videos]
    max_len = max(lengths)

    padded_videos = []
    for v in videos:
        pad_len = max_len - v.shape[0]
        if pad_len > 0:
            # pad with zeros at the end (on GPU)
            pad =  torch.zeros([pad_len, *v.shape[1:]], dtype=v.dtype, device=v.device)

            v = torch.cat([v, pad], dim=0)
        padded_videos.append(v.to(config.device))

    # stack for batch: shape → [B, T, C, H, W] (on GPU)
    padded_videos = torch.stack(padded_videos).to(config.device)

    # labels already have same shape → stack (on GPU)
    labels = torch.stack(labels).to(config.device)

    return padded_videos, labels, fnames


In [ ]:
!pip install vidgear


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
import cv2
from vidgear.gears import CamGear
from sklearn.model_selection import train_test_split
import sklearn.utils

# Dataset class
class SLDataset(Dataset):
    def __init__(self, df, video_transform: bool = True):
        self.df = sklearn.utils.shuffle(df).reset_index(drop=True)
        self.video_transform = video_transform

    def __getitem__(self, idx):
        if config.debug:
            print(f"Got item at index: {idx}")

        # ✅ get correct row from self.df (not global df!)
        video_path = self.df.iloc[idx, 1]
        encoding = torch.tensor(self.df.iloc[idx, 2])  # Keep on CPU initially
        enc_shape = encoding.shape[0]

        # ----- Read video according to selected method -----
        if config.video_processing_tool == 'OpenCV':
            reader = cv2.VideoCapture(video_path)

        elif config.video_processing_tool == 'VidGear':
            reader = CamGear(video_path).start()

        elif config.video_processing_tool == 'TorchVision':
            # Fix pts warning:
            reader, _, _ = torchvision.io.read_video(video_path, output_format="THWC", pts_unit="sec")

        # ----- Extract frames with hands -----
        hands_only = keep_frames_with_hands(reader, crop_size=600)  # (N, H, C, W) or similar

        # ✅ Ensure frames are detected
        if hands_only is None or hands_only.shape[0] == 0:
            if config.debug:
                print("Warning: No valid frames detected.")
            # Return zeros to avoid breaking DataLoader
            empty_tensor = torch.zeros((config.max_frames, 3, 224, 224), dtype=torch.float32, device=config.device)
            return empty_tensor, torch.reshape(encoding, (enc_shape, 1)).to(config.device), video_path

        # ✅ Fix shape to (N, C, H, W)
        if hands_only.ndim == 4 and hands_only.shape[1] not in [1, 3]:
            hands_only = hands_only.permute(0, 2, 1, 3)

        # ✅ Apply transforms once (on GPU)
        if self.video_transform:
            apply_trans = apply_video_transforms()
            hands_only = apply_trans(hands_only.to(config.device))

        if config.debug:
            print(f"Hands only shape: {hands_only.shape}")

        # ✅ Safely release readers
        if config.video_processing_tool == 'OpenCV':
            reader.release()
        elif config.video_processing_tool == 'VidGear':
            reader.stop()

        # ===== Frame count adjustment to config.max_frames =====
        n, c, h, w = hands_only.shape

        if n > config.max_frames and n < 2 * config.max_frames:
            left = (n - config.max_frames) // 2
            hands_only = hands_only[left:(n - left - 1)]

        elif n > config.max_frames:
            slice_step = ((n - 10) // config.max_frames + 1)
            hands_only = hands_only[5:(n - 5):slice_step]

        n = hands_only.shape[0]

        if n < config.max_frames:
            pad = hands_only[-(config.max_frames - n):]
            hands_only = torch.cat((hands_only, pad), dim=0)

        return hands_only.to(config.device), torch.reshape(encoding, (enc_shape, 1)).to(config.device), video_path

    def __len__(self):
        return len(self.df)

# Dataloader function
def get_dataloader(df, phase: str, batch_size: int = 96) -> DataLoader:
    # Split dataset into training and validation sets
    train_df, val_df = train_test_split(df, test_size=0.1,
                                        random_state=config.seed,
                                        stratify=df['idd'])
    train_df, val_df = train_df.reset_index(drop=True), val_df.reset_index(drop=True)

    # Choose the appropriate dataset (train/validation)
    chosen_df = train_df if phase == 'train' else val_df
    dataset = SLDataset(chosen_df, video_transform=True)

    # Create DataLoader with the chosen collate function
    dataloader = DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,  # Increased for faster CPU preprocessing
        collate_fn=pad_collate  # Ensure you have the collate function
    )
    return dataloader

# Example call
dl = get_dataloader(df, 'train', 1)


In [ ]:
import time

measure = time.time()
dl_next = next(iter(dl))
print('Data fetching:',time.time() - measure,'sec')

a,b,fname = dl_next

if config.debug:
  print(a.shape,b.shape,fname)
  visualize_frames(a)

In [ ]:
print(a.shape,b.shape,fname)
visualize_frames(a)

In [ ]:
# ========== CELL 24: EncoderRNN Class Definition ==========
import torchvision
from torchvision.models import squeezenet1_1, SqueezeNet1_1_Weights
from torchvision.models.feature_extraction import create_feature_extractor

# 🔧 CHANGED IN LAST CONVERSATION: Fixed EncoderRNN forward() and initHidden() for proper batch processing
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, device, biDirectional = False):
        super(EncoderRNN, self).__init__()

        model = squeezenet1_1(weights=SqueezeNet1_1_Weights.DEFAULT).to(config.device)
        return_nodes = {
            'features.12.cat': 'layer12'
        }
        self.pretrained_model = create_feature_extractor(model, return_nodes=return_nodes).to(config.device)
        self.pretrained_model.eval()

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.device = device
        self.D = 2 if biDirectional else 1

        self.rnn = nn.LSTM(
                input_size = self.input_size,
                hidden_size = self.hidden_size,  # Per-direction hidden size
                num_layers = 1,
                dropout = 0,
                bidirectional = biDirectional,
                batch_first = True).to(config.device)

    def forward(self, input, hidden):
        # 🔧 CHANGED: Fixed to handle batched 5D input [batch, frames, C, H, W]
        # input shape: (batch, frames, channels, height, width)
        batch_size, num_frames, C, H, W = input.shape

        # Reshape to (batch * frames, channels, height, width) for CNN
        input_reshaped = input.view(batch_size * num_frames, C, H, W)

        # Extract features
        with torch.no_grad():
            features = self.pretrained_model(input_reshaped)['layer12'].to(device=self.device)

        # features shape: (batch * frames, feature_channels, feat_h, feat_w)
        feat_channels, feat_h, feat_w = features.shape[1], features.shape[2], features.shape[3]

        # Reshape back to (batch, frames, features)
        features = features.view(batch_size, num_frames, feat_channels * feat_h * feat_w)

        output, hidden = self.rnn(features, hidden)
        return output, hidden

    def initHidden(self, batch_size=None):
        # 🔧 CHANGED: Made batch_size a parameter to handle variable batch sizes
        if batch_size is None:
            batch_size = config.BATCH_SIZE
        return (torch.zeros(self.D, batch_size, self.hidden_size, device=self.device),
                torch.zeros(self.D, batch_size, self.hidden_size, device=self.device))

In [ ]:
# ========== CELL 25: AttnDecoderRNN Class Definition ==========
class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, device, dropout_p=0.1, max_length=config.max_frames, biDirectional = False, debug=False): #max_length=config.max_words_in_sentence
        super(AttnDecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.dropout_p = dropout_p
        self.max_length = max_length
        self.debug = debug
        self.device = device

        self.D = 2 if biDirectional else 1

        if self.debug:
          print('Attn.init() hidden_size',hidden_size)
          print('Attn.init() output_size',output_size)
          print('Attn.init() max_length',max_length)

        self.embedding = nn.Embedding(self.output_size, self.hidden_size)
        self.attn = nn.Linear(self.hidden_size * 2, self.max_length)
        self.attn_combine = nn.Linear(self.hidden_size * 2, self.hidden_size)
        self.dropout = nn.Dropout(self.dropout_p)
        self.rnn = nn.LSTM(
                input_size = self.hidden_size,
                hidden_size = self.hidden_size,  # Per-direction hidden size
                num_layers = 1,
                dropout = 0,
                bidirectional = biDirectional,
                batch_first = True)
        self.out = nn.Linear(self.hidden_size * (2 if biDirectional else 1), self.output_size)

    def forward(self, input, hidden, encoder_outputs):
        embedded = self.embedding(input).view(input.shape[0],input.shape[1], self.hidden_size)
        embedded = self.dropout(embedded)
        if self.debug:
          print('Attn.forward() input',input.shape)
          print('Attn.forward() hidden',type(hidden),len(hidden),hidden[0].shape)
          print('Attn.forward() encoder_outputs',encoder_outputs.shape)
          print('embedded: ',embedded.shape)

        # 🔧 FIX: Handle batched input properly - use hidden[0][0] instead of hidden[0]
        attn_weights = F.softmax(self.attn(torch.cat((embedded[:, 0, :], hidden[0][0]), 1)), dim=1).to(device=self.device)
        attn_applied = torch.bmm(attn_weights.unsqueeze(1),encoder_outputs).to(device=self.device)

        output = torch.cat((embedded[:, 0, :], attn_applied[:, 0, :]), 1).to(device=self.device)
        output = self.attn_combine(output).unsqueeze(1).to(device=self.device)

        output = F.relu(output)
        output, hidden = self.rnn(output, hidden)

        output = F.log_softmax(self.out(output[:, 0, :]), dim=1).to(device=self.device)
        return output, hidden, attn_weights

    def initHidden(self):
        return (torch.zeros(self.D, 1, self.hidden_size*self.D, device=self.device),
                torch.zeros(self.D, 1, self.hidden_size*self.D, device=self.device))

In [ ]:
# ========== CELL 26: Training Function ==========
# 🔧 CHANGED IN LAST CONVERSATION: Added dynamic batch_size to fix encoder initialization
def train(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion, max_length=config.max_words_in_sentence):
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    if config.debug:
      print('Input len',input_tensor.shape,'Target len',target_tensor.shape)

    loss = 0

    # 🔧 CHANGED: Get actual batch size and pass to initHidden
    batch_size = input_tensor.size(0)
    encoder_hidden = encoder.initHidden(batch_size)
    encoder_output, encoder_hidden = encoder(input_tensor, encoder_hidden)

    # 🔧 FIX: Combine bidirectional encoder hidden states for unidirectional decoder
    # encoder_hidden is (h, c) where h is [2, batch, hidden] for bidirectional
    # decoder needs [1, batch, hidden*2] for unidirectional
    if encoder.D == 2:  # If encoder is bidirectional
        # Concatenate forward and backward hidden states
        h_combined = torch.cat((encoder_hidden[0][0:1], encoder_hidden[0][1:2]), dim=2)  # [1, batch, hidden*2]
        c_combined = torch.cat((encoder_hidden[1][0:1], encoder_hidden[1][1:2]), dim=2)  # [1, batch, hidden*2]
        decoder_hidden = (h_combined, c_combined)
    else:
        decoder_hidden = encoder_hidden

    decoder_input  = target_tensor[:,:(max_length-2),:]   # words from 1 to n-1
    decoder_target = target_tensor[:,1:(max_length-1),:]  # words from 2 to n (the target to the input word is the next word)
    tar_1hot = torch.nn.functional.one_hot(decoder_target, num_classes = len(encodings))

    if config.debug:
      print('Encoder hidden_0',len(encoder_hidden),'shape',encoder_hidden[0].shape)
      print('enc_out',encoder_output.shape)
      print('dec_in',decoder_input.shape)
      print('dec_target',decoder_target.shape)

    target_length = decoder_target.size(1)

    for di in range(target_length):
        if config.debug:
          print('dec hidden', decoder_hidden[0].shape,decoder_hidden[1].shape)

        decoder_output, decoder_hidden, decoder_attention = decoder(decoder_input[:,di,:], decoder_hidden, encoder_output)

        if config.debug:
          print('decoder_output & attn',decoder_output.shape, decoder_attention.shape)

        # 🔧 FIX: Handle batched loss calculation
        loss += criterion(decoder_output, tar_1hot[:, di, :].squeeze(1).float())

        if (decoder_target[:, di, 0] == encodings['EOS']).any().item():
          break

    loss.backward()

    encoder_optimizer.step()
    decoder_optimizer.step()

    if str(config.device).startswith('xla'):
      xm.mark_step()

    return loss.item() / (config.BATCH_SIZE*target_length)

In [ ]:
import time
from torch import optim
import torch.nn.functional as F
import gc

def trainIters(encoder, decoder, print_every=1000, plot_every=100, learning_rate=0.01):
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)

    criterion = nn.CrossEntropyLoss()

    trainloader = get_dataloader(df,'train',config.BATCH_SIZE)

    max_epochs = 1

    iter = 1
    start = time.time()
    for epoch in range(max_epochs):
      print('Starting epoch', epoch)
      for inputs, labels,fname in trainloader:
          if (iter%10 == 0):
            print('|', end = '')
          else:
            print('.', end = '')
          input_tensor = inputs.to(config.device)
          target_tensor = labels.to(config.device)

          try:
            loss = train(input_tensor, target_tensor, encoder,
                      decoder, encoder_optimizer, decoder_optimizer, criterion)
          except Exception as exp:
            print('There was an error: ',fname,exp)
            continue

          print_loss_total += loss
          plot_loss_total += loss

          if iter % print_every == 0:
              print_loss_avg = print_loss_total / print_every
              print_loss_total = 0
              print('%.4f' % (print_loss_avg))

              # model_scripted = torch.jit.script(encoder) # Export to TorchScript
              # model_scripted.save('/jamal/encoder.model') # Save
              # model_scripted = torch.jit.script(decoder) # Export to TorchScript
              # model_scripted.save('/jamal/decoder.model') # Save
              print('Time spent in seconds:',time.time() - start)
              start = time.time()

              torch.save(encoder.state_dict(),config.drive_folder+'/jamal/encoder_' + str(config.device) + '.model')
              torch.save(decoder.state_dict(),config.drive_folder+'/jamal/decoder_' + str(config.device) + '.model')

              gc.collect()

              if str(config.device).startswith('cuda'):
                torch.cuda.empty_cache()

          if iter % plot_every == 0:
              plot_loss_avg = plot_loss_total / plot_every
              plot_losses.append(plot_loss_avg)
              plot_loss_total = 0

          iter += 1

    #showPlot(plot_losses)

In [ ]:
import time
from torch import optim
import torch.nn.functional as F
import gc

def trainIters(encoder, decoder, print_every=1000, plot_every=100, learning_rate=0.01):
    plot_losses = []
    print_loss_total = 0
    plot_loss_total = 0

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)

    criterion = nn.CrossEntropyLoss()

    trainloader = get_dataloader(df, 'train', config.BATCH_SIZE)
    max_epochs = 15

    iter = 1
    global_start = time.time()

    total_batches = len(trainloader)

    for epoch in range(max_epochs):
        print(f"\n\n===== 🚀 Starting Epoch {epoch+1}/{max_epochs} =====")
        epoch_start = time.time()

        for batch_idx, (inputs, labels, fname) in enumerate(trainloader, start=1):

            batch_start = time.time()

            input_tensor = inputs.to(config.device)
            target_tensor = labels.to(config.device)

            try:
                loss = train(
                    input_tensor, target_tensor, encoder, decoder,
                    encoder_optimizer, decoder_optimizer, criterion
                )
            except Exception as exp:
                print(f"\n⚠️ Error on file {fname}: {exp}")
                continue

            # Update tracking
            print_loss_total += loss
            plot_loss_total += loss

            # --------- BATCH-LEVEL LOGGING ----------
            print(f"\n[Epoch {epoch+1}/{max_epochs}] "
                  f"[Batch {batch_idx}/{total_batches}] "
                  f"Loss: {loss:.4f}  "
                  f"Time: {(time.time() - batch_start):.2f}s")

            # ETA
            batch_time = time.time() - epoch_start
            avg_batch_time = batch_time / batch_idx
            remaining_batches = total_batches - batch_idx
            eta = remaining_batches * avg_batch_time

            print(f"   → ETA to finish epoch: {eta:.1f}s")

            # ---------- PRINT EVERY N ----------
            if iter % print_every == 0:
                print_avg_loss = print_loss_total / print_every
                print_loss_total = 0

                print(f"\n📌 PRINT_EVERY STEP ({iter}) → Avg Loss: {print_avg_loss:.4f}")

                # Save model
                enc_path = f"{config.drive_folder}/jamal/encoder_{str(config.device)}.model"
                dec_path = f"{config.drive_folder}/jamal/decoder_{str(config.device)}.model"

                torch.save(encoder.state_dict(), enc_path)
                torch.save(decoder.state_dict(), dec_path)

                print("💾 Models saved.")
                print(f"⏱ Time since last print: {time.time() - global_start:.2f}s")

                global_start = time.time()

                gc.collect()
                if str(config.device).startswith('cuda'):
                    torch.cuda.empty_cache()

            # ----------- PLOT EVERY N -----------
            if iter % plot_every == 0:
                plot_avg_loss = plot_loss_total / plot_every
                plot_losses.append(plot_avg_loss)
                plot_loss_total = 0

            iter += 1

        # END OF EPOCH SUMMARY
        epoch_time = time.time() - epoch_start
        print(f"\n🎉 Finished Epoch {epoch+1}/{max_epochs} in {epoch_time:.2f}s")
        print(f"🔥 Avg batch time: {epoch_time / total_batches:.2f}s")

    # showPlot(plot_losses)
    print("\n===== TRAINING COMPLETE =====")


In [ ]:
input_size = 86528
hidden_size = 4
config.debug = False

encoder = EncoderRNN(input_size, hidden_size, device=config.device, biDirectional = True).to(config.device)
attn_decoder = AttnDecoderRNN(hidden_size*2, len(encodings), device=config.device, dropout_p=0.1, biDirectional = False, debug=config.debug).to(config.device)

# use the previous weights
#encoder.load_state_dict(torch.load(config.drive_folder+'/jamal/encoder_' + str(config.device) + '.model', map_location=torch.device(config.device)))
#attn_decoder.load_state_dict(torch.load(config.drive_folder+'/jamal/decoder_' + str(config.device) + '.model', map_location=torch.device(config.device)))

config.BATCH_SIZE = 4
trainIters(encoder, attn_decoder, print_every=50)

TRAIN ITERS WITH LOGS

## ⚠️ IMPORTANT FIX FOR Cell 29

In Cell 29, you need to change **ONE LINE** in the EncoderRNN class:

**Find this line (around line 1065 in Cell 29):**
```python
self.rnn = nn.LSTM(input_size=self.input_size, hidden_size=self.hidden_size * self.D, num_layers=1, dropout=0, bidirectional=biDirectional, batch_first=True).to(config.device)
```

**Change it to:**
```python
self.rnn = nn.LSTM(input_size=self.input_size, hidden_size=self.hidden_size, num_layers=1, dropout=0, bidirectional=biDirectional, batch_first=True).to(config.device)
```

**The change:** Remove `* self.D` from `hidden_size=self.hidden_size * self.D`

This fixes the "Expected hidden[0] size (2, 4, 8), got [2, 4, 4]" error.


## ⚡️ Data Pipeline GPU Optimization
- All tensor transforms (resize, crop, stacking, padding) are now performed on GPU as soon as possible after conversion.
- Increased `num_workers` in DataLoader for faster CPU preprocessing.
- For best GPU utilization, use the largest batch size that fits in memory.
- Note: Video reading and MediaPipe hand detection still run on CPU (no GPU support in these libraries).
- For further speedup, consider using GPU-accelerated video libraries if available.
